# Multi-Domain Geometric Model — Architecture Documentation

**Authors:** Vínicius Teixeira de Carvalho Freitas  

---

This notebook is a **comprehensive, implementation-level reference** for the multi-domain geometric event representation model developed in this project.  Every design decision is explained from first principles; mathematical derivations are followed by the exact Python/PyTorch code that realises them.

The model realises the decomposition

$$\varphi(e) = \Psi\bigl(\varphi_{\text{who}}(e),\; \varphi_{\text{where}}(e),\; \varphi_{\text{when}}(e)\bigr)$$

where each $\varphi_\star$ is a **domain-specific geometric encoder** and $\Psi$ is a **fusion mechanism** that may operate in a geometry-aware manner.

### Table of Contents
1. [Notebook Setup](#setup)
2. [Multi-Domain Framework](#framework)
3. [Actor Domain (WHO/WHOM)](#actor)
4. [Geo Domain (WHERE)](#geo)
5. [Temporal Domain (WHEN)](#temporal)
6. [Fusion Mechanisms](#fusion)
7. [Model Composition](#model)
8. [YAML Config System & Ablation Design](#yaml)


## 1. Notebook Setup <a id='setup'></a>

In [ ]:
import sys, math
from pathlib import Path

# Make the project root importable
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D

torch.manual_seed(0)
device = 'cpu'
print('PyTorch', torch.__version__)


## 2. Multi-Domain Framework <a id='framework'></a>

A GDELT event record $e$ bundles four distinct information types:

| Dimension | Raw columns | Domain encoder |
|-----------|-------------|----------------|
| **WHO / WHOM** | `Actor1*`, `Actor2*` (8 categorical attrs each) | $\varphi_{\text{who}}$ — graph neural network on co-occurrence graph |
| **WHERE** | `ActionGeo_Lat`, `ActionGeo_Long`, `ActionGeo_CountryCode` | $\varphi_{\text{where}}$ — (hyper)spherical encoder |
| **WHEN** | `Day` (YYYYMMDD integer) | $\varphi_{\text{when}}$ — product-manifold temporal encoder |
| **WHAT** | `QuadClass` (4-class CAMEO label) | target $y \in \{0,1,2,3\}$ |

The composite encoder is

$$\varphi(e) = \Psi\bigl(\varphi_{\text{who}}(e),\; \varphi_{\text{where}}(e),\; \varphi_{\text{when}}(e)\bigr) \in \mathbb{R}^{C}$$

where $C$ is the number of classes and $\Psi$ (the fusion head) maps the concatenated (or attention-weighted) domain representations to logits.  This decomposition gives the model **orthogonal inductive biases**: relational topology (actor graph), spherical geometry (spatial), and circular periodicity (temporal).

The architecture is entirely **config-driven**: swapping `model.actor.type`, `model.geo.type`, `model.temporal.type`, or `model.fusion.type` in a YAML file changes the inductive bias without touching any Python code.  This enables systematic ablation studies.


### 2.1 Inductive biases in deep learning

An **inductive bias** is any assumption a learning algorithm uses to generalise from a finite training set to unseen inputs.  Without inductive biases a neural network must memorise the training set; with the right ones it extracts transferable structure.

Classical examples:

| Architecture | Inductive bias | Encoded assumption |
|-------------|---------------|--------------------|
| CNN | Translation equivariance | Relevant patterns appear at any position |
| RNN/Transformer | Sequential order | Dependencies decay with temporal distance |
| GNN | Permutation invariance + locality | Relational structure matters; node identity does not |
| **This model** | Relational + spherical + circular | Events have three independent geometric structure types |

#### Why geometric inductive biases?

An MLP applied to raw features $(\text{lat}, \text{lon}, \text{doy}, \text{dow}, \text{actor\_id})$ can, in principle, learn any function of those inputs — but it must *discover* periodicity, spherical topology, and relational co-occurrence from data alone.  Encoding these structures directly into the architecture means:

1. **Fewer parameters** needed to represent the right function class.
2. **Better sample efficiency**: geometric constraints act as regularisers.
3. **Out-of-distribution robustness**: a model that knows geography is on a sphere will not be confused by coordinates near the date line ($\pm 180°$).

The three proposed biases decompose the event space into **orthogonal axes** — each encoder is responsible for exactly one type of structure, with no overlap in the raw features they consume.

#### Geometric deep learning perspective

Bronstein et al. (2021) propose a unified view of deep learning as learning functions that respect the **symmetry group** of the input domain.  For a feature space that is a manifold $\mathcal{M}$, the natural function class consists of maps that are *equivariant* (or *invariant*) to the isometries of $\mathcal{M}$.

- **Actor graph**: the isometry group is the symmetric group $S_n$ (node permutations). GNNs are exactly the permutation-equivariant function class.  
- **Geography on $S^2$**: the isometry group is $SO(3)$ (rotations). A hyperspherical encoder that operates via log/exp maps is equivariant to rotations applied consistently to all coordinates.  
- **Circular time on $S^1 \times S^1$**: the isometry group is $SO(2) \times SO(2)$ (cyclic shifts of annual and weekly cycles). A `SphericalLinear` on $(\sin\theta, \cos\theta)$ is equivariant to shifts in $\theta$ because it operates in the tangent space of the circle.


In [ ]:
# Structural overview — no heavy dependencies needed
from src.models.multi_domain.model import MultiDomainGeometricModel

# Minimal config — just to show the shape of the config dict
sample_cfg = {
    'actor':    {'type': 'sage_gnn',         'feat_embed_dim': 4, 'hidden_dim': 16, 'out_dim': 8, 'num_layers': 2, 'dropout': 0.1},
    'geo':      {'type': 'euclidean',         'out_dim': 4, 'hidden_dim': 8},
    'temporal': {'type': 'product_manifold',  'out_dim': 4, 'hidden_dim': 8},
    'fusion':   {'type': 'concat_mlp',        'hidden_dim': 16, 'dropout': 0.1},
}

model = MultiDomainGeometricModel(
    model_cfg=sample_cfg,
    actor_cardinalities=[5, 5, 5, 5, 5, 5, 5, 5],  # 8 categorical actor attrs
    geo_country_cardinality=50,
    num_classes=4,
)

total_params = sum(p.numel() for p in model.parameters())
print(f'Actor encoder    : {sum(p.numel() for p in model.actor_encoder.parameters()):>7,} params')
print(f'Geo encoder      : {sum(p.numel() for p in model.geo_encoder.parameters()):>7,} params')
print(f'Temporal encoder : {sum(p.numel() for p in model.temporal_encoder.parameters()):>7,} params')
print(f'Fusion head      : {sum(p.numel() for p in model.fusion.parameters()):>7,} params')
print(f'Total            : {total_params:>7,} params')


## 3. Actor Domain (WHO/WHOM) <a id='actor'></a>

### 3.1 Actor identity construction

Each GDELT event involves two actors. GDELT represents each actor with **8 categorical columns** (e.g. `Actor1Code`, `Actor1Name`, `Actor1CountryCode`, `Actor1Type1Code`, …).  We collapse these into a single string identifier by concatenating the 8 values with a `-` separator:

```
actor_id = f"{Actor1Code}-{Actor1Name}-{Actor1CountryCode}-{Actor1Type1Code}-..."
```

This compound key uniquely identifies the **full actor profile**, not just the name or country code alone.  Two events share an actor node if and only if all 8 attributes are identical.

The construction is vectorised:

```python
cols = [df[f"{prefix}Code"], df[f"{prefix}Name"], ...]
actor_ids = cols[0].str.cat(cols[1:], sep='-').values   # pandas vectorised join
```


### 3.1b  Message passing as a relational inductive bias

#### Why a graph, not a table?

GDELT stores actor attributes in flat relational columns.  A plain MLP on actor attributes treats each event independently — it never sees that the same pair (USA-GOVT, RUS-GOVT) appeared in 3 000 training events.  A GNN, by contrast, propagates information along co-occurrence edges, so **frequently co-occurring actors end up with similar embeddings** regardless of whether they share categorical attributes.

#### Message-passing neural networks (MPNN)

All three graph-based actor encoders are instances of the MPNN framework (Gilmer et al., 2017).  Layer $l$ updates node $v$ via:

$$\mathbf{m}_v^{(l)} = \text{AGGREGATE}\bigl(\{\mathbf{h}_u^{(l-1)} : u \in \mathcal{N}(v)\}\bigr)$$

$$\mathbf{h}_v^{(l)} = \text{UPDATE}\bigl(\mathbf{h}_v^{(l-1)},\; \mathbf{m}_v^{(l)}\bigr)$$

The three encoders differ in AGGREGATE and UPDATE:

| Encoder | AGGREGATE | UPDATE | Inductive bias |
|---------|-----------|--------|----------------|
| `sage_gnn` | Mean pooling | Concat + Linear | Uniform neighbourhood influence |
| `gat_gnn` | Attention-weighted sum | Linear (heads averaged) | Selective focus on relevant co-actors |
| `weighted_gnn` | Degree-normalised weighted sum | Linear | Frequent co-actors more influential |
| `attribute_only` | — | MLP on attributes | No relational context (fully inductive) |

#### Expressive power and the Weisfeiler-Leman test

Xu et al. (2019) proved that **MPNNs are at most as expressive as the 1-WL graph isomorphism test**: two nodes with identical $k$-hop neighbourhood structures will receive identical embeddings.  For node *classification* (not isomorphism), this suffices in practice — the relevant question is whether the neighbourhood structure is discriminative for the target label, which it is here: actors involved in Verbal Conflict events have structurally different co-occurrence graphs than those in Material Cooperation events.

#### Why 2 layers?

With $L=2$ layers, each actor's embedding integrates information from its 2-hop neighbourhood — i.e., actors that share a co-actor.  Deeper networks risk **over-smoothing**: all actors converge to the same embedding as every node becomes reachable.  The co-occurrence graph is typically dense (high-frequency actors like USA-GOVT are hubs), making over-smoothing a real risk beyond $L = 3$.


In [ ]:
# Illustrate actor ID construction from raw GDELT columns
import pandas as pd

fake_event = pd.DataFrame({
    'Actor1CountryCode':    ['US',  'RS'],
    'Actor1KnownGroupCode': ['',    ''],
    'Actor1EthnicCode':     ['',    ''],
    'Actor1Religion1Code':  ['',    ''],
    'Actor1Religion2Code':  ['',    ''],
    'Actor1Type1Code':      ['GOV', 'GOV'],
    'Actor1Type2Code':      ['',    ''],
    'Actor1Type3Code':      ['',    ''],
})

prefix = 'Actor1'
attr_suffixes = ['CountryCode','KnownGroupCode','EthnicCode','Religion1Code','Religion2Code','Type1Code','Type2Code','Type3Code']
cols = [fake_event[f"{prefix}{s}"].fillna('').astype(str) for s in attr_suffixes]
actor_ids = cols[0].str.cat(cols[1:], sep='-').values

for eid, aid in enumerate(actor_ids):
    print(f'Event {eid}: actor_id = "{aid}"')


### 3.2 Actor co-occurrence graph

The **actor co-occurrence graph** $G = (\mathcal{V}, \mathcal{E})$ is built from the training split only:

- **Nodes** $\mathcal{V}$: every unique actor appearing in the **training split only**.  Val/test actors not seen in training are mapped to index 0 (unknown actor) at inference time.
- **Edges** $\mathcal{E}$: $(u, v) \in \mathcal{E}$ iff actors $u$ and $v$ co-occurred in at least one **training** event.  Test-split co-occurrences are not added.
- **Node features** $X \in \mathbb{Z}^{|\mathcal{V}| \times 8}$: label-encoded categorical columns per actor.
- **Edge weights** (used by `weighted_gnn` only): normalised co-occurrence count

$$w_{uv} = \frac{\text{count}(u,v)}{\max_{(i,j) \in \mathcal{E}} \text{count}(i,j)}$$

#### Inductive setting

The graph is built on the training split **only** (edges).  An actor can appear in val/test without ever having a training edge.  In that case `actor_to_idx.get(id, 0)` maps it to the **zero node** (index 0), which receives a zero embedding.  The `attribute_only` encoder is fully inductive: it ignores the graph and derives representations purely from node attribute embeddings.

In [ ]:
# Conceptual illustration of the graph construction logic
import numpy as np

# Simulated training events with actor pairs
train_pairs = [
    ('USA-GOVT', 'RUS-GOVT'),
    ('USA-GOVT', 'CHN-GOVT'),
    ('USA-GOVT', 'RUS-GOVT'),   # repeat -> higher edge weight
    ('CHN-GOVT', 'BRA-GOVT'),
]
all_actors = sorted({a for pair in train_pairs for a in pair})
actor_to_idx = {a: i+1 for i, a in enumerate(all_actors)}  # 0 = unknown
print('Nodes:', actor_to_idx)

from collections import Counter
edge_counts = Counter()
for a1, a2 in train_pairs:
    key = tuple(sorted([actor_to_idx[a1], actor_to_idx[a2]]))
    edge_counts[key] += 1

max_count = max(edge_counts.values())
print('\nEdges (src, dst, normalised_weight):')
for (u, v), cnt in edge_counts.items():
    print(f'  {u} -- {v}: count={cnt}, weight={cnt/max_count:.3f}')


### 3.3 Node feature embeddings

Each of the 8 categorical columns has a different vocabulary size (cardinality).  We learn a separate `nn.Embedding` for each attribute:

$$\mathbf{x}_v = [\mathbf{e}_1(x_v^{(1)}) \| \cdots \| \mathbf{e}_8(x_v^{(8)})] \in \mathbb{R}^{8 \cdot d_e}$$

where $d_e$ = `feat_embed_dim` (default 16) and $\|$ denotes concatenation.  Index 0 is reserved for **unknown / missing** actors (`padding_idx=0`) and receives a zero embedding.  The cardinalities are computed from the training split by `actor_graph_builder` and passed to the encoder factory.


In [ ]:
from src.models.multi_domain.actor_encoders import _ActorEncoderBase

# 3 attributes, each with vocabulary size 10, embedding dim 4
cardinalities = [10, 10, 10]
feat_embed_dim = 4

base = _ActorEncoderBase(cardinalities, feat_embed_dim)

# Node feature matrix: 5 nodes, each with 3 categorical attributes
x = torch.tensor([[0, 0, 0],    # unknown (padding)
                   [1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9],
                   [2, 3, 1]], dtype=torch.long)

emb = base._encode_node_features(x)
print(f'Input  shape: {x.shape}         (N_nodes, num_attrs)')
print(f'Output shape: {emb.shape}  (N_nodes, num_attrs * feat_embed_dim)')
print(f'Node 0 (unknown) all zeros: {emb[0].abs().sum().item():.4f}')


### 3.4 Actor encoders

All four actor encoders share the `_ActorEncoderBase` embedding layer and a `pair_proj` MLP.  They differ in whether and how they perform **message passing** over the co-occurrence graph.

#### 3.4.1 SAGEConv (default)

GraphSAGE aggregates neighbor features by mean pooling:

$$\mathbf{h}_v^{(l+1)} = \sigma\!\left( W^{(l)} \cdot \left[ \mathbf{h}_v^{(l)} \| \frac{1}{|\mathcal{N}(v)|} \sum_{u \in \mathcal{N}(v)} \mathbf{h}_u^{(l)} \right] \right)$$

No edge weights; every neighbour contributes equally.  Default depth is 2 layers (each actor "sees" its 2-hop neighbourhood).

#### 3.4.2 GATConv

Multi-head attention computes per-edge coefficients:

$$\alpha_{uv}^{(k)} = \text{softmax}_u \left( \text{LeakyReLU}\left( \mathbf{a}^{(k)\top} [W^{(k)} \mathbf{h}_u \| W^{(k)} \mathbf{h}_v] \right) \right)$$

With `concat=False`, head outputs are averaged: $\mathbf{h}_v^{(l+1)} = \sigma\!\left(\frac{1}{K}\sum_k \sum_u \alpha_{uv}^{(k)} W^{(k)} \mathbf{h}_u\right)$.  This keeps the output dimension stable at `hidden_dim` regardless of head count.

#### 3.4.3 GCNConv with edge weights

$$\mathbf{h}_v^{(l+1)} = \sigma\!\left( \sum_{u \in \mathcal{N}(v) \cup \{v\}} \frac{w_{uv}}{\sqrt{\tilde{d}_u \tilde{d}_v}} W^{(l)} \mathbf{h}_u^{(l)} \right)$$

Edge weights $w_{uv}$ are the **normalised co-occurrence counts** from the training split.  Actor pairs that co-occur more frequently in training events exert more influence during aggregation.

#### 3.4.4 Attribute-only (fully inductive)

No message passing.  Each actor is represented purely by its attribute embeddings:

$$\mathbf{z}_v = \text{MLP}(\mathbf{x}_v)$$

This is the only encoder that can represent **actors never seen during training** without a fallback to the zero node.

### 3.5 Pair representation

After message passing, we have one embedding per node.  An event involves an ordered pair $(a_1, a_2)$.  We **concatenate** the two actor embeddings and project through a shared MLP:

$$\varphi_{\text{who}}(e) = \text{MLP}_{\text{pair}}([\mathbf{h}_{a_1} \| \mathbf{h}_{a_2}]) \in \mathbb{R}^{d_{\text{actor}}}$$

The MLP includes a `LayerNorm`, `ReLU`, and `Dropout` for regularisation.  Crucially, **message passing is run once on the full graph**, and then `actor1_idx` / `actor2_idx` index into the resulting node embedding matrix — so the cost of propagation is $O(|\mathcal{E}|)$ per batch, not $O(B \cdot |\mathcal{E}|)$.


In [ ]:
from src.models.multi_domain.actor_encoders import ActorSAGEEncoder, ActorGATEncoder
from src.models.multi_domain.actor_encoders import ActorWeightedEncoder, ActorAttributeEncoder

cardinalities = [10, 10, 10, 10, 10, 10, 10, 10]  # 8 attrs, vocab size 10 each
N_nodes = 6

# Minimal random graph
graph_x = torch.randint(0, 10, (N_nodes, 8))
edge_index = torch.tensor([[0,1,2,3,4,5,1,2,3,4,5,0],
                            [1,2,3,4,5,0,0,1,2,3,4,5]], dtype=torch.long)
edge_attr  = torch.rand(edge_index.shape[1])

# Two event actor pairs
actor1_idx = torch.tensor([0, 2])
actor2_idx = torch.tensor([3, 5])

for EncoderClass, kwargs, name in [
    (ActorSAGEEncoder,      {}, 'SAGEConv'),
    (ActorGATEncoder,       {'gat_heads': 2}, 'GATConv'),
    (ActorWeightedEncoder,  {}, 'GCN+weights'),
    (ActorAttributeEncoder, {}, 'Attribute-only'),
]:
    common = dict(cardinalities=cardinalities, feat_embed_dim=4,
                  hidden_dim=16, out_dim=8, dropout=0.0)
    if name != 'Attribute-only':
        common['num_layers'] = 2
    enc = EncoderClass(**common, **kwargs).eval()
    with torch.no_grad():
        out = enc(graph_x, edge_index, actor1_idx, actor2_idx, graph_edge_attr=edge_attr)
    print(f'{name:20s}: output shape = {out.shape}')


## 4. Geo Domain (WHERE) <a id='geo'></a>

### 4.1 From coordinates to $S^2$

Latitude and longitude are **angles** on the Earth's surface.  The natural representation is a point on the unit 2-sphere $S^2 \subset \mathbb{R}^3$.  We convert via the standard geodetic-to-Cartesian map:

$$\phi_{S^2}(\text{lat}, \text{lon}) = \begin{pmatrix} \cos(\text{lat}) \cos(\text{lon}) \\ \cos(\text{lat}) \sin(\text{lon}) \\ \sin(\text{lat}) \end{pmatrix} \in S^2$$

where lat, lon are in radians.  By construction $\|\phi_{S^2}(\text{lat}, \text{lon})\|_2 = 1$.  **Geographic proximity is reflected as angular proximity**: cities on the same continent have a smaller great-circle angle between them than cities on opposite sides of the globe.


### 4.1b  Why the sphere is the right space for geographic coordinates

#### The problem with raw degrees in MLPs

Consider two events near the international date line:

| Event | Lat | Lon | Euclidean distance |
|-------|-----|-----|-------------------|
| Samoa | $-14°$ | $-171°$ | — |
| Tonga | $-21°$ | $+175°$ | $\sqrt{7^2 + 346^2} \approx 346$ |

The MLP sees $\Delta\text{lon} = 346°$ — a huge number — yet the two islands are geographically adjacent (actual distance $\approx 900$ km, same tectonic plate).  After projection to $S^2$, both map to nearby points with a great-circle angle of $\approx 8°$.  An MLP on raw degrees must *learn* to ignore this artefact; a spherical encoder never creates it.

#### Geodesic distance vs Euclidean distance

On the 2-sphere, the **geodesic** (shortest path on the surface) between two points $x, y \in S^2$ is:

$$d_{\text{geo}}(x, y) = \arccos(x \cdot y) \in [0, \pi]$$

The Euclidean distance in $\mathbb{R}^3$ satisfies $d_{\text{Eucl}}(x, y) = 2 \sin(d_{\text{geo}}/2)$, which is a monotone function of the geodesic distance.  Both are preserved by the projection — so a dot-product similarity (cosine similarity) in $\mathbb{R}^3$ is a *monotone function of true geographic proximity*.  No MLP on raw $(\text{lat}, \text{lon})$ has this property.

#### The equatorial projection and latitude distortion

Another subtlety: a degree of longitude near the equator covers $\approx 111$ km, but at latitude $60°N$ it covers only $\approx 55$ km.  An MLP treating $\Delta\text{lon}$ uniformly over all latitudes embeds a systematic geographic distortion.  The $S^2$ projection naturally accounts for this via the $\cos(\text{lat})$ factor in the Cartesian coordinates.

#### From $S^2$ to $S^{d-1}$: why go deeper?

The projection to $S^2 \subset \mathbb{R}^3$ captures the raw geographic structure.  However, $S^2$ is only 2-dimensional — it cannot represent higher-level geographic concepts (geopolitical regions, climate zones, conflict zones).  The `SphericalLinear` layers map $S^2 \to S^{d-1}$ for $d \gg 3$, learning a **higher-dimensional spherical embedding** where geopolitically similar regions cluster together regardless of their physical distance.  Crucially, every intermediate representation still lives on a sphere, so the inductive bias is maintained throughout — the network cannot collapse to a flat Euclidean space at any layer.


In [ ]:
import torch
import numpy as np

def to_s2(lat_deg, lon_deg):
    lat = lat_deg * (np.pi / 180)
    lon = lon_deg * (np.pi / 180)
    return np.array([np.cos(lat)*np.cos(lon), np.cos(lat)*np.sin(lon), np.sin(lat)])

def great_circle_deg(p1, p2):
    dot = np.clip(np.dot(p1, p2), -1, 1)
    return np.degrees(np.arccos(dot))

places = {
    'Sao Paulo':   (-23.5, -46.6),
    'Rio de Jan.': (-22.9, -43.2),
    'Buenos Aires':(-34.6, -58.4),
    'London':      (51.5,   -0.1),
    'Sydney':      (-33.9, 151.2),
}
s2 = {name: to_s2(lat, lon) for name, (lat, lon) in places.items()}

print('Great-circle angles (degrees):')
names = list(places.keys())
for i, n1 in enumerate(names):
    for n2 in names[i+1:]:
        print(f'  {n1:20s} <-> {n2:20s}: {great_circle_deg(s2[n1], s2[n2]):.1f} deg')


### 4.2 Riemannian geometry on $S^{d-1}$

A hypersphere $S^{d-1} = \{x \in \mathbb{R}^d : \|x\|_2 = 1\}$ is a smooth Riemannian manifold.  The key operations are the **exponential** and **logarithmic maps**, which let us move between the manifold and its tangent space.

#### Tangent space at the north pole

We choose the **north pole** $p = (0, \ldots, 0, 1) \in S^{d-1}$ as the reference point.  The tangent space there is

$$T_p S^{d-1} = \{v \in \mathbb{R}^d : v_d = 0\} \cong \mathbb{R}^{d-1}$$

i.e. all vectors whose **last coordinate is zero**.  This is the key invariant that `log_north` / `exp_north` preserve.

#### Log map at $p$

For a point $x \in S^{d-1}$ with $x \neq p$, let $\theta = \arccos(x_d)$ be the geodesic distance from $p$.  Then

$$\log_p(x) = \frac{\theta}{\|x_{1:d-1}\|} \cdot (x_{1:d-1}, 0)$$

The last coordinate is identically 0, consistent with $T_p S^{d-1}$.  When $x = p$ (i.e. $\theta = 0$) the map returns the zero vector.

#### Exp map at $p$

For a tangent vector $v \in T_p S^{d-1}$ (so $v_d = 0$) with $\|v\| = r$:

$$\exp_p(v) = \left( \frac{\sin r}{r} v_{1:d-1},\; \cos r \right)$$

This is the great-circle starting at $p$ in direction $v$, evaluated at arc-length $r$.  The output always satisfies $\|\exp_p(v)\|_2 = 1$.


In [ ]:
from src.models.multi_domain.riemannian import log_north, exp_north

# Verify log_north o exp_north = identity (should recover x)
torch.manual_seed(42)
N, d = 5, 4
x_raw = torch.randn(N, d)
x = F.normalize(x_raw, p=2, dim=-1)   # points on S^3

v   = log_north(x)                     # tangent vectors (last coord = 0)
x_r = exp_north(v)                     # map back

print('=== log_north properties ===')
print(f'Last coord of v (should be 0): {v[:, -1].abs().max().item():.2e}')

print('\n=== exp_north o log_north = id ===')
err = (x - x_r).norm(dim=-1)
print(f'Max reconstruction error: {err.max().item():.2e}')

print('\n=== exp_north output stays on S^{d-1} ===')
norms = exp_north(v).norm(dim=-1)
print(f'Output norms: {[round(n.item(),5) for n in norms]}')


In [ ]:
# Visual: log/exp map on S^1 (a circle)
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: points on S^1 and their log_north images (tangent axis)
ax = axes[0]
theta_vals = np.linspace(-np.pi + 0.15, np.pi - 0.15, 9)
for th in theta_vals:
    x_pt = np.array([np.sin(th), np.cos(th)])   # S^1, north pole = (0,1)
    xt   = torch.tensor(x_pt, dtype=torch.float32).unsqueeze(0)
    vt   = log_north(xt)[0].numpy()              # tangent vector (v_1, 0)
    ax.plot(*x_pt, 'bo', ms=5)
    va = 'bottom' if x_pt[1] > 0 else 'top'
    ax.annotate(f'{int(round(np.degrees(th)))}deg', x_pt, fontsize=7, ha='center', va=va)
    ax.annotate('', xy=(x_pt[0]*0.55, x_pt[1]*0.55),
                xytext=(vt[0]*0.25, 0.9),
                arrowprops=dict(arrowstyle='->', color='red', lw=0.8))

circle = plt.Circle((0, 0), 1, fill=False, color='gray', lw=1)
ax.add_patch(circle)
ax.plot(0, 1, 'r*', ms=12, label='north pole p')
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.4, 1.4); ax.set_aspect('equal')
ax.legend(fontsize=8); ax.set_title('S^1: log_north maps points -> tangent axis')

# Right: exp_north maps tangent vectors back to S^1
ax2 = axes[1]
v_range = np.linspace(-2.8, 2.8, 300)
ax2.plot(np.sin(v_range), np.cos(v_range), 'gray', lw=1)
ax2.axhline(1, color='orange', lw=1, ls='--', label='tangent at p')

for v_val in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    vt = torch.tensor([[v_val, 0.0]], dtype=torch.float32)
    pt = exp_north(vt)[0].numpy()
    ax2.plot(*pt, 'bo', ms=7)
    ax2.annotate(f'v={v_val:.0f}', pt, fontsize=8, ha='center')
    ax2.annotate('', xy=pt, xytext=(v_val*0.35, 1.0),
                 arrowprops=dict(arrowstyle='->', color='red', lw=0.8))

ax2.plot(0, 1, 'r*', ms=12)
ax2.set_xlim(-1.5, 1.5); ax2.set_ylim(-1.2, 1.3); ax2.set_aspect('equal')
ax2.legend(fontsize=8); ax2.set_title('S^1: exp_north maps tangent -> sphere')

plt.tight_layout(); plt.show()


### 4.3 `SphericalLinear` — intrinsic linear layer on $S^{d-1}$

`SphericalLinear(in_dim, out_dim)` maps $S^{\text{in}-1} \to S^{\text{out}-1}$ *without ever leaving the manifold*:

1. **Log**: $v = \log_p(x) \in T_p S^{\text{in}-1}$, drop last coord → $v_{1:\text{in}-1} \in \mathbb{R}^{\text{in}-1}$
2. **Linear**: $u = W v_{1:\text{in}-1} + b$, where $W \in \mathbb{R}^{(\text{out}-1) \times (\text{in}-1)}$
3. **Exp**: re-append zero → $\hat{u} = (u, 0)$, then $y = \exp_p(\hat{u}) \in S^{\text{out}-1}$

**Weight initialisation**: $W$ is initialised with `nn.init.orthogonal_`, which is an isometry in tangent space — it preserves geodesic distances early in training, stabilising gradient flow.

### 4.4 `SphereReLU` — nonlinearity on $S^{d-1}$

ReLU is applied in the tangent space at $p$:

$$\text{SphereReLU}(x) = \exp_p\bigl(\text{ReLU}(\log_p(x)_{1:d-1}),\; 0\bigr)$$

The tangent constraint ($v_d = 0$) is preserved because $\text{ReLU}(0) = 0$.  This is a **nonlinear map on the manifold**: directions that point away from $p$ with negative tangent coordinates are collapsed toward $p$, while positive-coordinate directions are preserved.


In [ ]:
from src.models.multi_domain.riemannian import SphericalLinear, SphereReLU

# 1. SphericalLinear: outputs always on unit sphere
torch.manual_seed(7)
sl = SphericalLinear(in_dim=3, out_dim=5).eval()

x_s2 = F.normalize(torch.randn(8, 3), p=2, dim=-1)  # 8 points on S^2
with torch.no_grad():
    y    = sl(x_s2)
print('SphericalLinear(3->5)')
print(f'  Input  shape: {x_s2.shape}  norms: {[round(n.item(),3) for n in x_s2.norm(dim=-1)]}')
print(f'  Output shape: {y.shape}  norms: {[round(n.item(),5) for n in y.norm(dim=-1)]}')

# 2. SphereReLU: also stays on sphere
print()
relu_s = SphereReLU()
with torch.no_grad():
    yr = relu_s(y)
print('SphereReLU')
print(f'  Input  norms: {[round(n.item(),5) for n in y.norm(dim=-1)]}')
print(f'  Output norms: {[round(n.item(),5) for n in yr.norm(dim=-1)]}')


### 4.5 Geo encoder implementations

| Type | Output lives on | Description |
|------|----------------|-------------|
| `hyperspherical` | $S^{d-1}$ | Fully Riemannian: $S^2 \to S^{h-1} \to S^{d-1}$, log/exp every step |
| `projected` | $S^{d-1}$ | $S^2 \to$ Euclidean MLP $\to$ L2-normalise (ablation baseline) |
| `euclidean` | $\mathbb{R}^d$ | Plain MLP on (lat, lon) — no sphere inductive bias |
| `region_aware` | $S^{d-1}$ | Riemannian path blended with country embedding in tangent space |

**`hyperspherical`** is the main proposed encoder.  The full pipeline:

```
(lat, lon)  -> _to_s2 -> S^2  -> SphericalLinear(3, hidden) -> S^{h-1}
                              -> SphereReLU                  -> S^{h-1}
                              -> SphericalLinear(h, out)     -> S^{out-1}
```

Every intermediate activation lies exactly on a hypersphere.  Gradients flow through geodesics (log/exp maps) rather than through arbitrary Euclidean paths.

**`projected`** replaces the Riemannian path with an unconstrained MLP; L2-normalisation forces the output onto $S^{d-1}$ but intermediate activations are Euclidean.  Comparing `projected` vs `hyperspherical` ablates the value of the Riemannian constraint on intermediate representations.

**`region_aware`** injects country-level context in the tangent space after the first spherical layer:

```
(lat, lon) -> S^2 -> SphericalLinear(3, h) -> SphereReLU -> S^{h-1}
               -> log_north -> v in R^{h-1}
country    -> Embedding -> r in R^{r_dim}
[v, r]     -> Linear(h-1+r, out-1)  -> exp_north -> S^{out-1}
```


In [ ]:
from src.models.multi_domain.geo_encoders import (
    HypersphericalEncoder, ProjectedEncoder,
    EuclideanEncoder, RegionAwareEncoder
)

# 6 random (lat, lon) pairs
lat_lon = torch.tensor([
    [-23.5, -46.6],   # Sao Paulo
    [ 48.9,   2.4],   # Paris
    [ 35.7, 139.7],   # Tokyo
    [ -1.3,  36.8],   # Nairobi
    [ 55.8,  37.6],   # Moscow
    [ 40.7, -74.0],   # New York
], dtype=torch.float32)
country_idx = torch.tensor([0, 1, 2, 3, 4, 5], dtype=torch.long)

configs = [
    (HypersphericalEncoder(out_dim=8, hidden_dim=16), None, 'hyperspherical'),
    (ProjectedEncoder(out_dim=8, hidden_dim=16),      None, 'projected     '),
    (EuclideanEncoder(out_dim=8, hidden_dim=16),      None, 'euclidean     '),
    (RegionAwareEncoder(out_dim=8, hidden_dim=16, region_cardinality=10, region_embed_dim=4),
     country_idx, 'region_aware  '),
]

print(f'{"Encoder":20s}  {"Shape":12s}  {"Unit norm?":10s}  max|norm-1|')
print('-'*60)
for enc, cidx, name in configs:
    enc.eval()
    with torch.no_grad():
        out = enc(lat_lon, geo_country_idx=cidx)
    norms = out.norm(dim=-1)
    on_sphere = (norms - 1).abs().max().item() < 1e-5
    print(f'{name:20s}  {str(out.shape):12s}  {str(on_sphere):10s}  {(norms-1).abs().max().item():.2e}')


## 5. Temporal Domain (WHEN) <a id='temporal'></a>

### 5.1 Raw temporal features

The dataset produces a **3-dimensional raw feature vector** per event:

$$\mathbf{t} = [t_{\text{linear}},\; \text{doy},\; \text{dow}] \in \mathbb{R}^3$$

- $t_{\text{linear}} = (\text{days since 2000-01-01} - \mu) / \sigma$ — z-scored on the training split.
- $\text{doy} \in [1, 366]$ — day of year (float).
- $\text{dow} \in [0, 6]$ — day of week (Mon = 0).

**Why raw?**  Keeping sin/cos computation out of the dataset means each encoder can impose its own temporal geometry.  A Fourier encoder learns data-driven frequencies; a product-manifold encoder uses fixed periods; a Riemannian encoder processes the circular components intrinsically on $S^1$.

### 5.2 Product manifold $\mathbb{R} \times S^1_{365} \times S^1_7$

The natural temporal structure is a **product manifold**:

- $\mathbb{R}$: linear progression of time (a Euclidean component).
- $S^1_{365}$: annual cycle — events near day 180 wrap around to day 1 of the next year.
- $S^1_7$: weekly cycle — Mondays are similar to other Mondays.

A plain MLP on raw $(t, \text{doy}, \text{dow})$ does not capture this periodicity without explicit sin/cos features.  The `product_manifold` encoder adds them:

$$\mathbf{f} = [t_{\text{linear}},\; \sin(2\pi \text{doy}/365),\; \cos(2\pi \text{doy}/365),\; \sin(2\pi \text{dow}/7),\; \cos(2\pi \text{dow}/7)] \in \mathbb{R}^5$$

### 5.3 Encoder implementations

| Type | Manifold output | Key idea |
|------|----------------|----------|
| `product_manifold` | $\mathbb{R}^d$ | Fixed sin/cos periods + MLP |
| `learnable_period` | $\mathbb{R}^d$ | Log-space learnable period parameters |
| `fourier` | $\mathbb{R}^d$ | Learnable frequency matrix $W \in \mathbb{R}^{K \times 3}$, $2K$ Fourier features |
| `riemannian_product` | $S^{d-1}$ | SphericalLinear on each $S^1$ component; blend in tangent space |



### 5.1b  Why circular geometry matters for temporal features

#### The aliasing problem in raw integer encoding

Consider two events: December 31 ($\text{doy} = 365$) and January 1 ($\text{doy} = 1$).  They are separated by one day, but their raw day-of-year values differ by 364.  An MLP on raw $\text{doy}$ must learn, from data, to treat these as similar — a difficult task since it requires the MLP to discover the circular topology of the calendar year from event labels alone.

The standard fix is to replace $\text{doy}$ with its sine-cosine embedding:

$$\phi_{\text{annual}}(\text{doy}) = \left(\sin\frac{2\pi\,\text{doy}}{365},\; \cos\frac{2\pi\,\text{doy}}{365}\right) \in S^1$$

Now $\|\phi_{\text{annual}}(365) - \phi_{\text{annual}}(1)\|_2 \approx 0$ — Dec 31 and Jan 1 are correctly adjacent.  The same applies to the weekly cycle with period 7.

#### Why sin/cos alone may not be enough

Fixed sin/cos embeddings capture **one frequency per cycle** (the fundamental).  But real-world event patterns may have harmonics: a bi-annual pattern has a dominant frequency at period 182.5, electoral cycles depend on the country's calendar.  The `LearnablePeriodEncoder` addresses this by treating the periods as free parameters:

$$T_{\text{annual}} = e^{\lambda_{\text{annual}}}, \quad T_{\text{weekly}} = e^{\lambda_{\text{weekly}}}$$

The log-space parameterisation keeps $T > 0$ throughout training.  The `FourierEncoder` goes further by learning a **frequency matrix** $W \in \mathbb{R}^{K \times 3}$ that discovers the $K$ most informative frequencies from data:

$$\phi_{\text{Fourier}}(\mathbf{t}) = [\mathbf{t},\; \sin(W\mathbf{t}),\; \cos(W\mathbf{t})] \in \mathbb{R}^{3+2K}$$

#### The product manifold structure $\mathbb{R} \times S^1 \times S^1$

Time has a richer structure than any single manifold:

- **Linear component** $t_{\text{linear}} \in \mathbb{R}$: captures long-term trend (events in 2020 may differ structurally from 1995).
- **Annual cycle** $\text{doy} \in S^1$: captures seasonal effects (harvest conflicts, monsoon floods, winter diplomacy).
- **Weekly cycle** $\text{dow} \in S^1$: captures institutional rhythms (UN votes on Tuesdays, press releases on Mondays).

The **product manifold** $\mathbb{R} \times S^1 \times S^1$ is the minimal space that represents all three structures simultaneously without conflation.  A joint MLP on all three raw features sees one 3-dimensional vector and must discover internally that two of its dimensions are circular — the product manifold encoder makes this explicit.

#### What `RiemannianProductEncoder` adds

The `product_manifold` encoder applies a fixed linear map from $\mathbb{R}^5$ to $\mathbb{R}^{d_{\text{out}}}$ — it can represent any linear combination of the 5 sin/cos + linear features.  The `RiemannianProductEncoder` replaces the fixed single-frequency $S^1$ embedding with a **SphericalLinear**, which is equivalent to learning a bank of frequencies for each circular component:

$$\text{SphericalLinear}(2, h):\quad S^1 \to S^{h-1}$$

This maps each point on the circle to a point on a $(h-1)$-dimensional sphere, learning $h-1$ independent frequencies (via the tangent-space linear map).  The output space $S^{h-1}$ has strictly more capacity than the 2-dimensional sin/cos representation — it can encode multi-frequency periodic structure while maintaining the circular inductive bias at every layer.


In [ ]:
from src.models.multi_domain.temporal_encoders import (
    ProductManifoldEncoder, LearnablePeriodEncoder,
    FourierEncoder, RiemannianProductEncoder
)

# Simulate 6 events across different dates: [t_linear, doy, dow]
time_features = torch.tensor([
    [-1.2, 15.0, 0.0],   # Jan 15, Monday
    [ 0.0, 180.0, 3.0],  # Jun 29, Thursday
    [ 1.5, 355.0, 6.0],  # Dec 21, Sunday
    [ 0.3,  90.0, 1.0],  # Apr 01, Tuesday
    [-0.8, 270.0, 4.0],  # Sep 27, Friday
    [ 2.1,  1.0,  2.0],  # Jan 01, Wednesday
], dtype=torch.float32)

print(f'{"Encoder":25s}  {"Output shape":14s}  {"Unit norm?"}')
print('-'*55)
for EncoderClass, kwargs, name in [
    (ProductManifoldEncoder,   {},                     'product_manifold   '),
    (LearnablePeriodEncoder,   {},                     'learnable_period   '),
    (FourierEncoder,           {'num_frequencies': 8}, 'fourier (K=8)      '),
    (RiemannianProductEncoder, {},                     'riemannian_product '),
]:
    enc = EncoderClass(out_dim=8, hidden_dim=16, **kwargs).eval()
    with torch.no_grad():
        out = enc(time_features)
    norms = out.norm(dim=-1)
    on_sphere = (norms - 1).abs().max().item() < 1e-5
    print(f'{name:25s}  {str(out.shape):14s}  {on_sphere}')


### 5.4 `RiemannianProductEncoder` — manifold-aware temporal encoding

This encoder treats each circular component as a point on $S^1$ and processes it with a `SphericalLinear`:

$$S^1 \ni (\sin\theta, \cos\theta) \xrightarrow{\text{SphericalLinear}(2, h)} S^{h-1}$$

A `SphericalLinear(2, h)` on $(\sin\theta, \cos\theta) \in S^1$ is equivalent to learning a **bank of $h-1$ independent frequencies** for $\theta$, then mapping the result to $S^{h-1}$ via the exp map.  This strictly generalises fixed-frequency sin/cos.

The three components are blended in **tangent space** of the output sphere:

```
doy  -> (sin, cos) in S^1 -> SphericalLinear(2, h) -> z_ann in S^{h-1}
dow  -> (sin, cos) in S^1 -> SphericalLinear(2, h) -> z_wk  in S^{h-1}
t    -> Linear(1, t_dim)  -> z_t in R^{t_dim}

v_ann = log_north(z_ann)[..., :-1]  in R^{h-1}
v_wk  = log_north(z_wk) [..., :-1]  in R^{h-1}

u_tan = Linear([v_ann, v_wk, z_t])  in R^{out-1}
output = exp_north(u_tan, 0)        in S^{out-1}
```

where `t_dim = max(hidden_dim // 4, 1)` and `blend_in = 2*(hidden_dim-1) + t_dim`.

The linear time component is embedded in Euclidean space and injected in the tangent blend, giving the encoder a smooth handle on both absolute time and circular periodicities.


In [ ]:
# Demonstrate that RiemannianProductEncoder captures annual periodicity
enc = RiemannianProductEncoder(out_dim=4, hidden_dim=8).eval()

# All events: same year (t=0), same weekday (dow=0), varying day-of-year
doy_vals = torch.linspace(1, 365, 37)
t_feats  = torch.stack([
    torch.zeros_like(doy_vals),   # t_linear = 0
    doy_vals,                     # doy
    torch.zeros_like(doy_vals),   # dow = Monday
], dim=1)

with torch.no_grad():
    z = enc(t_feats)  # (37, 4) on S^3

# Great-circle angle between each consecutive day pair
cos_sim = F.cosine_similarity(z[:-1], z[1:], dim=-1)
angles  = torch.acos(cos_sim.clamp(-1, 1)) * (180 / 3.14159)

plt.figure(figsize=(8, 3))
plt.plot(doy_vals[:-1].numpy(), angles.numpy())
plt.xlabel('Day of year (start of interval)')
plt.ylabel('Great-circle angle (deg) between consecutive days')
plt.title('RiemannianProductEncoder: angular change along the year')
plt.tight_layout()
plt.show()


## 6. Fusion Mechanisms <a id='fusion'></a>

The fusion module $\Psi$ receives a list of domain embeddings

$$[\mathbf{z}_{\text{who}},\; \mathbf{z}_{\text{where}},\; \mathbf{z}_{\text{when}}]$$

and outputs class logits $\in \mathbb{R}^C$.  Four strategies are implemented.

### 6.1 `concat_mlp` — late concatenation (default)

$$\Psi_{\text{concat}}(z_1, z_2, z_3) = \text{MLP}([z_1 \| z_2 \| z_3])$$

Every view contributes equally to the classifier input.  Simplest and fastest; serves as the main baseline.

### 6.2 `attention` — attention-weighted fusion

A shared query vector $q$ scores each view:

$$\alpha_i = \text{softmax}_i\bigl(\tanh(W_i z_i) \cdot q\bigr), \quad i \in \{\text{who, where, when}\}$$

$$\Psi_{\text{attn}} = \text{MLP}\!\left( \sum_i \alpha_i V_i z_i \right)$$

The model can dynamically weight domains per event — useful if geography is more informative for natural disasters while actors matter more for diplomatic events.

### 6.3 `gated` — sigmoid gating

A global context vector (concat of all views) generates per-unit sigmoid gates:

$$g = \sigma(W_g [z_1 \| z_2 \| z_3]), \quad \Psi_{\text{gated}} = \text{MLP}\!\left(g \odot \sum_i V_i z_i\right)$$

Soft suppression: zero-filled coordinates (e.g. missing geolocation) can be gated out without hard thresholds.

### 6.4 `geometry_aware` — Riemannian log-map preprocessing

When some views live on a hypersphere ($S^{d-1}$) and others in Euclidean space ($\mathbb{R}^d$), directly concatenating them mixes **incompatible geometries**:

- Sphere vectors are unit-norm by construction — bounded $\ell_2$ contribution.
- Euclidean GNN outputs have no norm constraint — potentially large scale.

The `geometry_aware` wrapper resolves this by mapping each sphere view to the **tangent space at the north pole** before fusion:

$$z_{\text{where}}^{\prime} = \log_p(z_{\text{where}})[_{1:d-1}] \in \mathbb{R}^{d-1}$$

The tangent displacement is geometrically meaningful: it is the **geodesic vector** pointing from $p$ toward the encoded location.  Its magnitude encodes how far the point is from $p$; its direction encodes the manifold direction.  After log-mapping, all views are in a flat Euclidean space where standard MLPs and attention mechanisms make sense.

Note the dimension change: $d$-dimensional sphere output → $(d-1)$-dimensional tangent coords.  The inner fusion module (concat_mlp / attention / gated) is sized with the **effective dimensions**:

$$d_{\text{eff}} = \begin{cases} d - 1 & \text{if view is on } S^{d-1} \\ d & \text{otherwise} \end{cases}$$

The manifold type per view is **auto-detected** from encoder type strings in `model.py` — no manual annotation in YAML is required.


### 6.0  The geometry of mixed-manifold fusion

#### Scale mismatch between sphere and Euclidean views

After encoding, the three domain views have heterogeneous geometric properties:

| View | Space | Norm constraint | Typical $\|z\|_2$ |
|------|-------|-----------------|--------------------|
| Actor (GNN) | $\mathbb{R}^{d_a}$ | None | $O(\sqrt{d_a})$ after LayerNorm |
| Geo (sphere) | $S^{d_g - 1}$ | $\|z_g\|_2 = 1$ always | 1.0 (exact) |
| Temporal (sphere) | $S^{d_t - 1}$ | $\|z_t\|_2 = 1$ always | 1.0 (exact) |

Naively concatenating $[z_a \| z_g \| z_t] \in \mathbb{R}^{d_a + d_g + d_t}$ and feeding this to a Linear layer creates an implicit **scale imbalance**: the gradient $\partial \mathcal{L}/\partial z_g$ is bounded by the weight-row norm, but $\partial \mathcal{L}/\partial z_a$ is not — the GNN output can dominate the fusion head's signal.  Normalising $z_a$ post-hoc is ad-hoc; the log map provides a principled alternative.

#### The log map as a local linearisation

For a sphere view $z \in S^{d-1}$, the Riemannian log map at the north pole $p$ gives:

$$v = \log_p(z)_{1:d-1} \in \mathbb{R}^{d-1}, \quad \|v\|_2 = \theta = d_{\text{geo}}(p, z)$$

This is the **geodesic displacement** from $p$ to $z$.  Its norm is exactly the great-circle distance from the reference point, ranging in $[0, \pi]$.  After log-mapping, the sphere view becomes a $(d-1)$-dimensional Euclidean vector with a natural magnitude — comparable to typical GNN output norms after LayerNorm.

#### Curvature and the tangent-space approximation

A sphere $S^{d-1}$ has **constant positive sectional curvature** $K = 1$.  The tangent space $T_p S^{d-1}$ is a flat (zero-curvature) Euclidean space that is the best linear approximation of the manifold near $p$.  For points close to $p$ (small $\theta$), the log map is nearly isometric: $d_{\text{Eucl}}(\log_p(x), \log_p(y)) \approx d_{\text{geo}}(x, y)$.  For points far from $p$ (large $\theta$), the approximation degrades, but the MLP/attention head can learn to correct for this distortion.

The choice of $p = (0, \ldots, 0, 1)$ (north pole) is arbitrary — any point would give a valid tangent space — but the north pole simplifies the log/exp map formulas to closed form and is a natural default.

#### Dimension reduction: $d \to d-1$

A point on $S^{d-1} \subset \mathbb{R}^d$ has $d-1$ degrees of freedom (one is consumed by the unit-norm constraint).  The log map recovers exactly these $d-1$ free coordinates: the last component of the tangent vector at $p$ is identically zero and is dropped.  This means the `geometry_aware` fusion head operates on a slightly smaller input than a naive `concat_mlp` would — fewer parameters, no redundant dimension.

#### Geometry-aware vs geometry-blind: when does it matter?

The log-map preprocessing matters most when:
1. The sphere views encode information with high angular variance (points spread broadly over $S^{d-1}$).
2. The Euclidean view has very different magnitude than 1.
3. The fusion head is shallow (less capacity to compensate for the scale mismatch internally).

The ablation configs `fusion_ga_concat` and `fusion_ga_attention` (geometry_aware with concat_mlp and attention inner types, respectively) allow isolating whether the log-map preprocessing, or the gated inner fusion, is responsible for any performance gain.


In [ ]:
from src.models.multi_domain.fusion import (
    ConcatMLPFusion, AttentionFusion, GatedFusion,
    GeometryAwareFusion, build_fusion
)

# Simulate 3 domain embeddings: actor (Euclidean), geo (sphere), temporal (sphere)
B = 4
z_actor = torch.randn(B, 8)                      # Euclidean, d=8
z_geo   = F.normalize(torch.randn(B, 6), dim=-1)  # on S^5, d=6
z_time  = F.normalize(torch.randn(B, 4), dim=-1)  # on S^3, d=4

view_dims      = [8, 6, 4]
view_manifolds = ['euclidean', 'sphere', 'sphere']
num_classes    = 4

print(f'{"Fusion type":20s}  {"Logits shape":12s}  {"Params":>8s}  Notes')
print('-'*70)
for cfg, notes in [
    ({'type': 'concat_mlp',     'hidden_dim': 16, 'dropout': 0.0},
     'views=[8,6,4], cat=[18]->MLP'),
    ({'type': 'attention',      'hidden_dim': 16, 'dropout': 0.0, 'attention_dim': 8},
     'dot-product scores per view'),
    ({'type': 'gated',          'hidden_dim': 16, 'dropout': 0.0},
     'sigmoid gate from context'),
    ({'type': 'geometry_aware', 'inner_type': 'concat_mlp', 'hidden_dim': 16, 'dropout': 0.0},
     'log-map S^5->R^5, S^3->R^3 first'),
]:
    fusion = build_fusion(cfg, view_dims, num_classes, view_manifolds).eval()
    with torch.no_grad():
        logits = fusion([z_actor, z_geo, z_time])
    n_params = sum(p.numel() for p in fusion.parameters())
    print(f'{cfg["type"]:20s}  {str(logits.shape):12s}  {n_params:>8,}  {notes}')


In [ ]:
# Visualise the geometry_aware log-map: S^2 points -> tangent plane
import numpy as np
from src.models.multi_domain.riemannian import log_north

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(121, projection='3d')
ax2 = fig.add_subplot(122)

cities = {
    'Sao Paulo':   (-23.5, -46.6),
    'Paris':       (48.9,    2.4),
    'Tokyo':       (35.7,  139.7),
    'Nairobi':     (-1.3,   36.8),
    'New York':    (40.7,  -74.0),
    'Sydney':      (-33.9, 151.2),
}

def deg2s2(lat_d, lon_d):
    lat = np.radians(lat_d); lon = np.radians(lon_d)
    return np.array([np.cos(lat)*np.cos(lon), np.cos(lat)*np.sin(lon), np.sin(lat)])

colors = plt.cm.Set1(np.linspace(0, 0.8, len(cities)))

u_s, v_s = np.mgrid[0:2*np.pi:30j, 0:np.pi:15j]
xs = np.cos(u_s)*np.sin(v_s); ys = np.sin(u_s)*np.sin(v_s); zs = np.cos(v_s)
ax1.plot_wireframe(xs, ys, zs, alpha=0.08, color='gray', lw=0.4)
ax1.scatter(0, 0, 1, c='red', s=80, zorder=5, label='north pole p')

tangent_pts = []
for (name, (lat, lon)), col in zip(cities.items(), colors):
    pt = deg2s2(lat, lon)
    ax1.scatter(*pt, c=[col], s=60)
    ax1.text(*pt * 1.12, name, fontsize=6)
    pt_t = torch.tensor(pt, dtype=torch.float32).unsqueeze(0)
    v    = log_north(pt_t)[0].numpy()  # (3,) tangent vec, v[-1]=0
    tangent_pts.append((name, v[:2], col))

ax1.set_title('S^2 city locations')
ax1.legend(fontsize=7)

for name, v2d, col in tangent_pts:
    ax2.scatter(*v2d, c=[col], s=60)
    ax2.annotate(name, v2d, fontsize=8)

ax2.axhline(0, color='gray', lw=0.5)
ax2.axvline(0, color='gray', lw=0.5)
ax2.set_title('Tangent plane at north pole (after log_north)')
ax2.set_xlabel('v1 (geodesic E-W component)')
ax2.set_ylabel('v2 (geodesic N-S component)')

plt.tight_layout()
plt.show()


## 7. Model Composition <a id='model'></a>

### 7.1 `MultiDomainGeometricModel`

`MultiDomainGeometricModel` is the top-level `nn.Module`.  Its `__init__` builds all submodules from a nested YAML config dict:

```python
self.actor_encoder    = build_actor_encoder(actor_cfg, actor_cardinalities)
self.geo_encoder      = build_geo_encoder(geo_cfg, geo_country_cardinality)
self.temporal_encoder = build_temporal_encoder(temporal_cfg)
self.fusion           = build_fusion(fusion_cfg, view_dims, num_classes, view_manifolds)
```

**Manifold auto-detection**: the model infers `view_manifolds` from encoder type strings.  No manual annotation is needed in the YAML:

```python
_SPHERE_GEO_TYPES      = {'hyperspherical', 'region_aware', 'projected'}
_SPHERE_TEMPORAL_TYPES = {'riemannian_product'}
view_manifolds = [
    'euclidean',
    'sphere' if geo_cfg['type'] in _SPHERE_GEO_TYPES else 'euclidean',
    'sphere' if temporal_cfg['type'] in _SPHERE_TEMPORAL_TYPES else 'euclidean',
]
```

**Actor graph as buffers**: the actor co-occurrence graph is loaded via `model.set_actor_graph(...)` and registered as `torch.Tensor` buffers.  `model.to(device)` then moves all graph tensors to the target device in one call — no manual `.to(device)` needed in the runner.

### 7.2 Forward pass

```python
def forward(self, actor1_idx, actor2_idx, geo, time_features, geo_country_idx=None):
    z_actor = self.actor_encoder(
        self._graph_x, self._graph_edge_index,   # full graph (registered buffers)
        actor1_idx, actor2_idx,                   # per-event actor indices
        graph_edge_attr=self._graph_edge_attr
    )
    z_geo  = self.geo_encoder(geo, geo_country_idx=geo_country_idx)
    z_time = self.temporal_encoder(time_features)
    return self.fusion([z_actor, z_geo, z_time])   # -> (B, num_classes)
```

Message passing runs once per batch on the **full** training graph.  `actor1_idx` / `actor2_idx` then index into the resulting $(N_{\text{nodes}} \times h)$ embedding matrix.  Cost: $O(|\mathcal{E}| + B)$ rather than $O(B \cdot |\mathcal{E}|)$.


In [ ]:
from src.models.multi_domain.model import MultiDomainGeometricModel

torch.manual_seed(0)

# Full geometry stack: hyperspherical + riemannian_product + geometry_aware fusion
cfg = {
    'actor':    {'type': 'sage_gnn', 'feat_embed_dim': 4, 'hidden_dim': 16,
                 'out_dim': 8, 'num_layers': 2, 'dropout': 0.0},
    'geo':      {'type': 'hyperspherical', 'out_dim': 6, 'hidden_dim': 12},
    'temporal': {'type': 'riemannian_product', 'out_dim': 4, 'hidden_dim': 8},
    'fusion':   {'type': 'geometry_aware', 'inner_type': 'concat_mlp',
                 'hidden_dim': 32, 'dropout': 0.0},
}

model = MultiDomainGeometricModel(
    model_cfg=cfg,
    actor_cardinalities=[10]*8,
    geo_country_cardinality=20,
    num_classes=4,
).eval()

# Load actor graph
N_nodes = 8
graph_x = torch.randint(0, 10, (N_nodes, 8))
edge_index = torch.tensor([[0,1,2,3,4,5,6,7,1,2,3,4,5,6,7,0],
                            [1,2,3,4,5,6,7,0,0,1,2,3,4,5,6,7]], dtype=torch.long)
model.set_actor_graph(graph_x, edge_index)

# Batch of 4 events
batch = {
    'actor1_idx':      torch.tensor([0, 2, 4, 6]),
    'actor2_idx':      torch.tensor([1, 3, 5, 7]),
    'geo':             torch.tensor([[-23.5,-46.6],[48.9,2.4],[35.7,139.7],[-1.3,36.8]],
                                    dtype=torch.float32),
    'geo_country_idx': torch.tensor([0, 1, 2, 3]),
    'time_features':   torch.tensor([[-1.2,15.0,0.0],[0.0,180.0,3.0],
                                     [1.5,355.0,6.0],[0.3,90.0,1.0]],
                                    dtype=torch.float32),
    'labels':          torch.tensor([0, 1, 2, 3]),
}

with torch.no_grad():
    logits, labels = model.forward_batch(batch, device='cpu')

print(f'Logits shape: {logits.shape}  (B, num_classes)')
print(f'Predicted:  {logits.argmax(dim=-1).tolist()}')
print(f'True:       {labels.tolist()}')

print('\nParameter breakdown:')
for name, sub in [('actor_encoder', model.actor_encoder),
                   ('geo_encoder',   model.geo_encoder),
                   ('temporal_enc',  model.temporal_encoder),
                   ('fusion',        model.fusion)]:
    n = sum(p.numel() for p in sub.parameters())
    print(f'  {name:20s}: {n:6,} params')


### 7.3 `MultiDomainEventDataset` and `MultiDomainDataModule`

#### Dataset

`MultiDomainEventDataset` converts a GDELT split DataFrame into a `dict` of per-domain tensors:

| Key | Shape | Contents |
|-----|-------|----------|
| `actor1_idx` | `(B,)` int64 | Node index of Actor 1 in actor graph |
| `actor2_idx` | `(B,)` int64 | Node index of Actor 2 in actor graph |
| `geo` | `(B, 2)` float32 | `[lat, lon]` in degrees |
| `geo_country_idx` | `(B,)` int64 | Label-encoded country code |
| `time_features` | `(B, 3)` float32 | `[t_linear, doy, dow]` |
| `labels` | `(B,)` int64 | `QuadClass` target |

**Temporal normalisation**: `t_linear` is z-scored.  The mean and std are **fit on the training split** and passed to val/test datasets — preventing data leakage.

#### DataModule

`MultiDomainDataModule.setup()` orchestrates:

1. Load train/val/test parquets from `SPLITS_DATA/{dataset_name}/`.
2. Build actor co-occurrence graph (**train split only** for edges).
3. Build `country_to_idx` mapping (all splits; no label leakage from this).
4. Construct datasets with training temporal statistics.

After `setup()`, the runner accesses `dm.actor_graph`, `dm.actor_cardinalities`, and `dm.geo_country_cardinality` to build `MultiDomainGeometricModel`.  The actor graph is then loaded into the model via `model.set_actor_graph(...)` and `model.to(device)` moves everything — including the graph buffers — to the target device.


## 8. YAML Config System & Ablation Design <a id='yaml'></a>

Every experiment is specified by a single YAML file under `src/config/model_setup/multi_domain/`.  The config drives the factory functions (`build_actor_encoder`, `build_geo_encoder`, `build_temporal_encoder`, `build_fusion`) without any code changes.

### 8.1 Config structure

```yaml
training:
  batch_size: 512
  epochs: 200
  patience: 30        # early stopping on val f1_macro
  lr: 0.001
  weight_decay: 0.0001
  device: cuda
  monitor_metric: f1_macro

model:
  actor:
    type: sage_gnn          # sage_gnn | gat_gnn | weighted_gnn | attribute_only
    feat_embed_dim: 16
    hidden_dim: 128
    out_dim: 64
    num_layers: 2
    dropout: 0.2

  geo:
    type: hyperspherical    # hyperspherical | projected | euclidean | region_aware
    out_dim: 32
    hidden_dim: 64

  temporal:
    type: riemannian_product # product_manifold | learnable_period | fourier | riemannian_product
    out_dim: 16
    hidden_dim: 32

  fusion:
    type: geometry_aware    # concat_mlp | attention | gated | geometry_aware
    inner_type: gated       # (only used when type=geometry_aware)
    hidden_dim: 128
    dropout: 0.2
```

### 8.2 Ablation design

The 22 configs in `run_experiments.sh` form a structured one-factor-at-a-time ablation study:

| Ablation axis | Fixed | Varied | Config files |
|---------------|-------|--------|-------------|
| **Actor encoder** | geo=hyper, temporal=riem, fusion=ga_gated | actor type | `actor_gat`, `actor_weighted`, `actor_attribute_only` |
| **Geo encoder** | actor=sage, temporal=riem, fusion=ga_gated | geo type | `geo_projected`, `geo_euclidean`, `geo_region_aware` |
| **Temporal encoder** | actor=sage, geo=hyper, fusion=ga_gated | temporal type | `temporal_product_manifold`, `temporal_learnable_period`, `temporal_fourier` |
| **Fusion** | actor=sage, geo=hyper, temporal=riem | fusion type | `fusion_concat`, `fusion_attention`, `fusion_ga_concat`, `fusion_ga_attention` |
| **Geometry inductive bias** | — | ablate sphere components | `ablation_no_geometry`, `ablation_geo_sphere_only`, `ablation_temporal_sphere_only`, `ablation_full_riemannian_concat` |

The geometry ablations specifically test whether Riemannian geometry in geo, temporal, or both is responsible for any performance gain over Euclidean baselines:

| Config | Geo | Temporal | Fusion inner |
|--------|-----|----------|--------------|
| `ablation_no_geometry` | euclidean | product_manifold | concat_mlp |
| `ablation_geo_sphere_only` | **hyperspherical** | product_manifold | geometry_aware(concat) |
| `ablation_temporal_sphere_only` | euclidean | **riemannian_product** | geometry_aware(concat) |
| `ablation_full_riemannian_concat` | **hyperspherical** | **riemannian_product** | geometry_aware(concat) |

Comparing `ablation_full_riemannian_concat` vs `multi_domain_riemannian_aware` (which also uses gated inner fusion) isolates the contribution of the gating mechanism on top of the Riemannian pre-processing.


In [ ]:
# List all available configs
from pathlib import Path

config_dir = Path('../src/config/model_setup/multi_domain')
configs = sorted(config_dir.glob('*.yaml'))

print(f'Found {len(configs)} multi-domain configs:\n')
for c in configs:
    print(f'  {c.name}')


In [ ]:
# Load and parse a config to verify the factory round-trip
import yaml
from src.models.multi_domain.model import MultiDomainGeometricModel

yaml_path = Path('../src/config/model_setup/multi_domain/multi_domain_riemannian_aware.yaml')
with open(yaml_path) as f:
    full_cfg = yaml.safe_load(f)

model_cfg = full_cfg['model']
print('Config actor.type:      ', model_cfg['actor']['type'])
print('Config geo.type:        ', model_cfg['geo']['type'])
print('Config temporal.type:   ', model_cfg['temporal']['type'])
print('Config fusion.type:     ', model_cfg['fusion']['type'])
print('Config fusion.inner_type:', model_cfg['fusion'].get('inner_type', 'N/A'))

m = MultiDomainGeometricModel(
    model_cfg=model_cfg,
    actor_cardinalities=[10]*8,
    geo_country_cardinality=50,
    num_classes=4,
)
print(f'\nModel built OK: {sum(p.numel() for p in m.parameters()):,} parameters')


---

## Summary

The multi-domain geometric model introduces three orthogonal inductive biases:

1. **Relational (WHO)**: graph neural network on actor co-occurrence graph.  Actor representations are contextualised by their co-occurrence neighbourhood in training events.

2. **Spherical (WHERE)**: geographic coordinates are projected to $S^2$ and processed with intrinsic Riemannian operations (log/exp maps, SphericalLinear, SphereReLU).  Geographic proximity maps directly to angular proximity at every network layer.

3. **Circular-temporal (WHEN)**: the day-of-year and day-of-week are treated as points on $S^1$ and processed with `SphericalLinear` — a learned multi-frequency circular encoder.  Periodicity is built into the representation, not just the input features.

The **geometry-aware fusion** ensures that representations from different manifolds ($\mathbb{R}^d$ vs $S^{d-1}$) are first brought to a common tangent space via the Riemannian log map before any MLP or attention mechanism operates on them.  This prevents scale incompatibilities and gives the fusion head geometrically coherent inputs.

All design choices are controlled by YAML configuration, enabling systematic ablation without code changes.
